# Задача 3. Распространение и детектирование света

**Обоснование выбора класса для метода Пуассона:**
Метод, разыгрывающий число фотоэлектронов, должен принадлежать классу `Detector`. Квантовая эффективность `a` и сам процесс преобразования фотонов в электроны происходят на фотокатоде, который является частью детектора. Источник (`Source`) лишь испускает фотоны и не знает о свойствах детектора.

**Алгоритм распространения:**
Мы используем аналитическое решение пересечения луча и сферы. Для каждого фотона вычисляется точка пересечения с границей детектора. Затем проверяется, находится ли эта точка в пределах радиуса `r` какого-либо ФЭУ.

In [2]:
import numpy as np

class Detector:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.pmt_coords = self._generate_pmt_coordinates()

    def _generate_pmt_coordinates(self):
        indices = np.arange(0, self.N, dtype=float) + 0.5
        phi = np.arccos(1 - 2 * indices / self.N)
        theta = np.pi * (1 + 5**0.5) * indices
        x = self.R * np.cos(theta) * np.sin(phi)
        y = self.R * np.sin(theta) * np.sin(phi)
        z = self.R * np.cos(phi)
        return np.column_stack((x, y, z))

    def propagate_photons(self, origin, directions):
        """
        Определяет, в какие ФЭУ попадут фотоны.
        origin: np.array (3,) - координаты источника
        directions: np.array (M, 3) - направления фотонов (M штук)
        Возвращает: массив (N,) с количеством фотонов, попавших в каждый ФЭУ.
        """
        # Уравнение пересечения сферы и луча: |origin + t*dir|^2 = R^2
        # t^2 + 2t*(origin @ dir) + origin^2 - R^2 = 0
        b = np.dot(directions, origin)
        c = np.dot(origin, origin) - self.R**2

        # Берем положительный корень (фотон летит от источника вперед)
        discriminant = b**2 - c
        t = -b + np.sqrt(discriminant)

        # Координаты точек попадания на сферу (M, 3)
        hit_points = origin + t[:, np.newaxis] * directions

        # Считаем расстояния от точек попадания до центров всех ФЭУ (M, N)
        distances = np.linalg.norm(hit_points[:, np.newaxis, :] - self.pmt_coords[np.newaxis, :, :], axis=2)

        # Проверяем попадание (расстояние <= r) -> матрица True/False (M, N)
        hits = distances <= self.r

        # Суммируем по фотонам (axis=0) -> сколько фотонов попало в каждый ФЭУ (N,)
        photons_per_pmt = np.sum(hits, axis=0)

        return photons_per_pmt

    def detect_photons(self, photons_per_pmt):
        """
        Разыгрывает количество фотоэлектронов по распределению Пуассона.
        """
        return np.random.poisson(lam=photons_per_pmt * self.a)

class Source:
    def __init__(self, Y=10000.0):
        self.Y = Y

    def generate_photons(self, E):
        N_photons = int(E * self.Y)
        phi = np.random.uniform(0, 2 * np.pi, N_photons)
        cos_theta = np.random.uniform(-1, 1, N_photons)
        sin_theta = np.sqrt(1 - cos_theta**2)

        x = sin_theta * np.cos(phi)
        y = sin_theta * np.sin(phi)
        z = cos_theta

        return np.column_stack((x, y, z))

In [3]:
# Инициализация
detector = Detector(R=2.0, N=40, r=0.1, a=0.5)
source = Source(Y=10000)

E = 1.0 # МэВ
origin = np.array([0.0, 0.0, 0.0]) # Центр детектора

# 1. Генерируем фотоны
directions = source.generate_photons(E)

# 2. Распределяем их по ФЭУ
photons_per_pmt = detector.propagate_photons(origin, directions)

# 3. Разыгрываем фотоэлектроны
pes_per_pmt = detector.detect_photons(photons_per_pmt)

# 4. Считаем полный заряд
total_Q = np.sum(pes_per_pmt)

# 5. Считаем ожидаемое значение
Q_expected = detector.a * source.Y * E * detector.N * (detector.r**2) / (4 * detector.R**2)

print(f"Суммарный заряд (число ФЭ): {total_Q}")
print(f"Ожидаемое значение Q: {Q_expected:.2f}")
print(f"Отношение (симуляция / теория): {total_Q / Q_expected:.3f}")

Суммарный заряд (число ФЭ): 115
Ожидаемое значение Q: 125.00
Отношение (симуляция / теория): 0.920
